In [ ]:
from anndata import read_h5ad
adata=read_h5ad("update_adata_for_paper_09_13.h5ad")

In [ ]:
with rc_context({'figure.figsize': (10, 10)}):
     sc.pl.umap(adata,color=[ "diameter",], size=2, cmap='rocket',save="_diameter_umap.png" )

In [ ]:
with rc_context({'figure.figsize': (10, 10)}):
     sc.pl.umap(adata,color=[ "follicle_type",], size=2, cmap='rocket',save="follicle_type_umap.png" )

In [ ]:
# for single sample

import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize

identifier = "A1-3"
Width_kpx = 8.6
Height_kpx = 8.6
df = pd.read_csv(
    f"result download from website/segment/33422-2363-adult1_{identifier}_DAPI__7.5_mu_dilated_segmentation_cell_polygons.csv",
    header=None,
    names=["cell_id", "x", "y"]
)

geometries = []
cell_ids = []

for cell_id, group in df.groupby("cell_id"):
    coords = list(zip(group["x"], group["y"]))
    polygon = Polygon(coords)
    geometries.append(polygon)
    cell_ids.append(cell_id)

xenium_cell_polygons = gpd.GeoDataFrame({'cell_id': cell_ids, 'geometry': geometries})

xenium_cell_ids = set(xenium_cell_polygons["cell_id"].astype(str))
adata_cell_ids = set(adata.obs["cell_id"].astype(str))

common_cell_ids = xenium_cell_ids & adata_cell_ids
print(f"{len(common_cell_ids)} cell_id are present in both xenium_cell_polygons and adata.obs")

valid_polygons = []
valid_clusters = []

for idx, row in xenium_cell_polygons.iterrows():
    cell_id = str(row["cell_id"])
    geom = row.geometry
    
    if (
        cell_id in common_cell_ids
        and geom.is_valid
        and geom.geom_type == 'Polygon'
        and len(geom.exterior.coords) >= 3
    ):
        coords = list(geom.exterior.coords)
        valid_polygons.append(coords)
        valid_clusters.append(adata.obs.loc[cell_id, "leiden"])
    else:
        print(f"Skipped invalid or non-common cell_id: {cell_id}")

if len(valid_polygons) == 0:
    raise ValueError("No valid polygons available for plotting!")

fig = plt.figure(figsize=(Width_kpx, Height_kpx))
ax = plt.gca()
fig.patch.set_facecolor('black')
ax.set_facecolor('black')

plot_polygons(
    valid_polygons,
    ax=ax,
    array=list(map(ctype_to_code_map.get, valid_clusters)),
    edgecolor="none",
    linewidth=0,
    norm=Normalize(vmin=0, vmax=len(celltypes)),
    cmap=ctype_cmap,
)

for polygon_coords, cluster_value in zip(valid_polygons, valid_clusters):
    try:
        x = [coord[0] for coord in polygon_coords]
        y = [coord[1] for coord in polygon_coords]
        center_x = np.mean(x)
        center_y = np.mean(y)
        label = str(cluster_value)
        ax.text(
            center_x, center_y, label,
            fontsize=0.01,
            ha='center', va='center',
            color='white'
        )
    except Exception as e:
        print(f"Failed to add label: {str(e)}")

ax.set_xticks([])
ax.set_yticks([])
plt.savefig(f'mapping_cluster/{identifier}_new.png', dpi=900, bbox_inches='tight', pad_inches=0.1)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ==============================================================================
# 1. Prepare data
# ==============================================================================
mean_df = average_composition.loc[:, (slice(None), 'mean')]
mean_df.columns = mean_df.columns.droplevel(1)

sem_df = average_composition.loc[:, (slice(None), 'sem')]
sem_df.columns = sem_df.columns.droplevel(1)

# ==============================================================================
# 2. Plot settings
# ==============================================================================
follicle_type_palette = {
    'healthy': '#1f77b4',
    'type1':   '#ff7f0e',
    'type2':   '#2ca02c',
    'type3':   '#d62728'
}

leiden_clusters = mean_df.columns.tolist()
follicle_types = mean_df.index.tolist()

n_leiden = len(leiden_clusters)
n_follicle = len(follicle_types)
bar_width = 0.8 / n_follicle
x_positions = np.arange(n_leiden)

# ==============================================================================
# 3. Plot
# ==============================================================================
fig, ax = plt.subplots(figsize=(20, 8))

for i, f_type in enumerate(follicle_types):
    offset = (i - (n_follicle - 1) / 2) * bar_width
    bar_positions = x_positions + offset
    
    means = mean_df.loc[f_type].values
    sems = sem_df.loc[f_type].values
    
    ax.bar(
        bar_positions, 
        means, 
        width=bar_width * 0.9,
        label=f_type, 
        color=follicle_type_palette.get(f_type, 'gray'),
        yerr=sems,
        capsize=3
    )

# ==============================================================================
# 4. Significance annotation
# ==============================================================================
def add_significance_bar(ax, x1, x2, y, text, h=0.01):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c='black')
    ax.text((x1 + x2) * 0.5, y + h, text, ha='center', va='bottom', color='black')

y_max = plot_df['mean_proportion'].max()

for _, row in stats_df.iterrows():
    p_val = row['p_value']
    
    if p_val < 0.001:
        sig_text = '***'
    elif p_val < 0.01:
        sig_text = '**'
    elif p_val < 0.05:
        sig_text = '*'
    else:
        continue

    ref_group, comp_group = row['comparison'].split('_vs_')
    cell_type = row['cell_type']
    
    leiden_idx = leiden_clusters.index(cell_type)
    ref_idx = follicle_types.index(ref_group)
    comp_idx = follicle_types.index(comp_group)
    
    x1_offset = (ref_idx - (n_follicle - 1) / 2) * bar_width
    x2_offset = (comp_idx - (n_follicle - 1) / 2) * bar_width
    
    x1_pos = leiden_idx + x1_offset
    x2_pos = leiden_idx + x2_offset
    
    bar_height1 = mean_df.loc[ref_group, cell_type] + sem_df.loc[ref_group, cell_type]
    bar_height2 = mean_df.loc[comp_group, cell_type] + sem_df.loc[comp_group, cell_type]
    y_pos = max(bar_height1, bar_height2) + 0.05 * y_max
    
    add_significance_bar(ax, x1_pos, x2_pos, y_pos, sig_text)

# ==============================================================================
# 5. Format figure
# ==============================================================================
ax.set_title('Average Cell Type Composition by Follicle Type', fontsize=16)
ax.set_xlabel('Leiden Cell Type', fontsize=12)
ax.set_ylabel('Average Proportion', fontsize=12)

ax.set_xticks(x_positions)
ax.set_xticklabels(leiden_clusters, rotation=45, ha='right')

ax.legend(title='Follicle Type')
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(
    "cell_composition_by_follicle_type.pdf",
    dpi=300,
    bbox_inches='tight'
)
plt.show()


In [ ]:
# Run this cell in Jupyter (requires `adata` to exist in memory)
# Batch gene plotting - exactly following your original plotting logic

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, sem
from scipy import sparse
from collections import OrderedDict

# ---------------------------
# Helper: get expression vector for one gene (prefer adata.raw if present)
# ---------------------------
def get_gene_vector(adata, gene, use_raw=True):
    if use_raw and getattr(adata, "raw", None) is not None and gene in adata.raw.var_names:
        X = adata.raw[:, gene].X
    elif gene in adata.var_names:
        X = adata[:, gene].X
    else:
        raise KeyError(f"Gene {gene} not found in adata.var_names or adata.raw.var_names")
    if sparse.issparse(X):
        vec = np.asarray(X.todense()).ravel()
    else:
        vec = np.asarray(X).ravel()
    return vec

# ---------------------------
# Single-gene plotting function - EXACT copy of your working code
# ---------------------------
def plot_single_gene_points_box_bordered(
    adata,
    gene_name,
    output_dir="debug_single_gene_plots",
    group_col_ft1="follicle_type1",
    leiden_q1_col="leiden_q1",
    exclude_main={"other"},
    reference_main="healthy",
    use_raw=True,
    p_adjust='none',          # 'none' | 'per_leiden' | 'per_gene'
    point_size=6,             # point area (s)
    jitter_strength=0.3,      # jitter proportion
    group_sep=1.6,            # separation between leiden groups
    tick_fontsize=9,
    title_fontsize=10,
    fig_width=None,           
    fig_height=5,           
    margin_x=0.05,           
    margin_y=0.05,           
    save_pdf=True,
    verbose=True
):
    os.makedirs(output_dir, exist_ok=True)

    # metadata checks
    if group_col_ft1 not in adata.obs.columns:
        raise KeyError(f"adata.obs missing {group_col_ft1}")
    if leiden_q1_col not in adata.obs.columns:
        raise KeyError(f"adata.obs missing {leiden_q1_col}")

    # get expression - with error handling for missing genes
    try:
        vec_all = get_gene_vector(adata, gene_name, use_raw=use_raw)
    except KeyError as e:
        if verbose:
            print(f"Skipping {gene_name}: {e}")
        return None

    # metadata arrays
    ft1_vals_all = adata.obs[group_col_ft1].astype(str).values
    leiden_vals_all = adata.obs[leiden_q1_col].astype(str).values
    cell_main_all = np.array([s.split("_")[0] if "_" in s else s for s in ft1_vals_all], dtype=str)

    # mask exclude mains
    exclude_set = set(exclude_main) if exclude_main is not None else set()
    mask_keep = np.array([m not in exclude_set for m in cell_main_all], dtype=bool)
    if mask_keep.sum() == 0:
        raise ValueError("No cells left after excluding mains. Check exclude_main.")

    vec = vec_all[mask_keep]
    ft1_vals = ft1_vals_all[mask_keep]
    leiden_vals = leiden_vals_all[mask_keep]
    cell_main = cell_main_all[mask_keep]

    # build mains list
    ft1_to_main = {ft1: (ft1.split("_")[0] if "_" in ft1 else ft1) for ft1 in np.unique(ft1_vals)}
    all_main = sorted({ft1_to_main[ft1] for ft1 in ft1_to_main})
    preferred_main_order = ["healthy","type1","type2","type3"]
    available_main = [m for m in preferred_main_order if m in all_main] + [m for m in all_main if m not in preferred_main_order]

    # leiden order
    desired_leiden_q1_order = ['GC','iTC','ST','FIB','other']
    leiden_q1_present = [l for l in desired_leiden_q1_order if l in np.unique(leiden_vals)]
    if len(leiden_q1_present) == 0:
        leiden_q1_present = list(np.unique(leiden_vals))

    # aggregate: compute ft1-level means per leiden/main
    leiden_main_vals = {l: {m: [] for m in available_main} for l in leiden_q1_present}
    for l in leiden_q1_present:
        mask_l = (leiden_vals == l)
        ft1_in_l = np.unique(ft1_vals[mask_l])
        for ft1 in ft1_in_l:
            mask_ft1 = (ft1_vals == ft1) & (leiden_vals == l)
            vals = vec[mask_ft1]
            if vals.size == 0:
                continue
            ft1_mean = np.nanmean(vals)
            m = ft1_to_main.get(ft1, ft1)
            if m in leiden_main_vals[l]:
                leiden_main_vals[l][m].append(ft1_mean)

    # statistics: mannwhitneyu per leiden per main vs reference_main (ft1-level)
    p_records = []
    for l in leiden_q1_present:
        for m in available_main:
            if m == reference_main:
                p = np.nan
            else:
                arrA = np.array(leiden_main_vals[l].get(m, []))
                arrB = np.array(leiden_main_vals[l].get(reference_main, []))
                if arrA.size >= 1 and arrB.size >= 1:
                    try:
                        stat, p = mannwhitneyu(arrA, arrB, alternative='two-sided')
                    except Exception:
                        p = np.nan
                else:
                    p = np.nan
            p_records.append({"gene": gene_name, "leiden_q1": l, "compared_main": m, "ref_main": reference_main, "p": p})

    # p-value handling (p_adjust='none' => q=p and signif p<0.05)
    for rec in p_records:
        rec['q'] = rec['p']
        rec['signif'] = bool((not (rec['p'] is None or np.isnan(rec['p']))) and (rec['p'] < 0.05))

    # ---------- plotting ----------
   
    if fig_width is None:
        fig_width = max(8, len(leiden_q1_present) * 1.8)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=150)

    n_groups = len(available_main)
    n_x = len(leiden_q1_present)
 
    width = min(0.4, 0.7 / max(1, n_groups)) 
    x = np.arange(n_x) * group_sep

    np.random.seed(0)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    main_colors = {m: colors[i % len(colors)] for i, m in enumerate(available_main)}

    # compute global max for annotation placement
    all_vals = []
    for l in leiden_q1_present:
        for m in available_main:
            arr = np.array(leiden_main_vals[l].get(m, []))
            if arr.size > 0:
                all_vals.append(np.nanmax(arr))
    global_max = np.nanmax(all_vals) if len(all_vals) > 0 else 1.0

    for i, m in enumerate(available_main):
        for j, l in enumerate(leiden_q1_present):
            pos = x[j] + i*width - (width * n_groups) / 2.0 + width/2.0
            arr = np.array(leiden_main_vals[l].get(m, []))
            if arr.size == 0:
                continue
            
            
            median_val = np.nanmedian(arr)
            line_width = width * 0.7  
            ax.plot([pos - line_width/2, pos + line_width/2], 
                   [median_val, median_val], 
                   color=main_colors[m], 
                   linewidth=4, 
                   alpha=0.8,
                   zorder=2)
            
            # points (completely no edges) - cleanest for Illustrator
            jitter = (np.random.rand(arr.size) - 0.5) * width * jitter_strength
            x_points = np.full(arr.size, pos) + jitter
            # === Points with explicitly no edges ===
            ax.scatter(x_points, arr,
                       s=point_size,
                       alpha=0.95,
                       marker='o',
                       color=main_colors[m],
                       edgecolors='none',  
                       linewidths=0,       
                       zorder=3)

    # EXACT same axis settings as your working code
    ax.set_xticks(x)
    ax.set_xticklabels(leiden_q1_present, rotation=0, ha='center', fontsize=tick_fontsize)

    # ax.set_ylabel("expression (mean per ft1)", fontsize=10)
    # ax.set_title(f"{gene_name} — by {leiden_q1_col} (points=ft1; lines=median; ref={reference_main}; p_adjust={p_adjust})",
    #              fontsize=title_fontsize)

    from matplotlib.ticker import MaxNLocator, FormatStrFormatter
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax.tick_params(axis='y', labelsize=16)
    ax.spines['left'].set_linewidth(1.5)
    ax.tick_params(axis='y', width=1.5)

 
    ax.set_xticks([])
    ax.margins(x=margin_x, y=margin_y)
    

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    # NO legend as requested

    # significance stars
    for j, l in enumerate(leiden_q1_present):
        recs = [r for r in p_records if r['leiden_q1'] == l]
        rec_by_main = {r['compared_main']: r for r in recs}
        for i, m in enumerate(available_main):
            if m == reference_main:
                continue
            rec = rec_by_main.get(m, None)
            if rec is None:
                continue
            q = rec.get('q', np.nan)
            if np.isnan(q):
                continue
            if q < 0.001:
                stars = '***'
            elif q < 0.01:
                stars = '**'
            elif q < 0.05:
                stars = '*'
            else:
                stars = ''
            if stars != '':
                pos = x[j] + i*width - (width * n_groups) / 2.0 + width/2.0
                arr = np.array(leiden_main_vals[l].get(m, []))
                ymax = np.nanmax(arr) if arr.size>0 else 0.0
                ytext = ymax + 0.03 * (global_max if not np.isnan(global_max) else 1.0)
                ax.text(pos, ytext, stars, ha='center', va='bottom', fontsize=16)

    plt.tight_layout()
    if save_pdf:
        out_file = os.path.join(output_dir, f"{gene_name}_debug_points_median_lines.pdf")
        fig.savefig(out_file, dpi=200, bbox_inches='tight')
        if verbose:
            print(f"Saved PDF: {out_file}")

    plt.show()
    plt.close(fig)

    # save p_records table
    stats_df = pd.DataFrame(p_records)
    stats_out = os.path.join(output_dir, f"{gene_name}_stats_p_adjust_{p_adjust}.csv")
    stats_df.to_csv(stats_out, index=False)
    if verbose:
        print(f"Saved stats CSV: {stats_out}")

    return stats_df

# ---------------------------
# Define your pathways
# ---------------------------
pathways = OrderedDict([
    ("Validation_Cell_Type_Markers", ["FSHR","ANPEP","ACTA2","KIT","PECAM1"]),
    ("Mesenchymal_Cell_Markers", ["ITGB1","CD44","THY1","NGFR"]),
    ("Steroidogenesis_Pathway", ["STAR","CYP17A1","CYP11A1","CYP19A1","HSD17B1"]),
    ("Hormone_Receptors", ["LHCGR","AR"]),
    ("NOTCH_Signalling", ["NOTCH1","NOTCH3","JAG1"]),
    ("WNT_Signalling", ["WNT6","FZD6"]),
    ("TGFb_BMP_Signalling", ["INHA","INHBA","INHBB","TGFBR1","ACVR1B","BMPR1B","BMPR2","ID1"]),
    ("HH_Signalling", ["DHH","IHH","PTCH1","PTCH2","GLI1","GLI2"])
])

# ---------------------------
# Batch processing - exactly following your parameter settings
# ---------------------------
base_output_dir = "pathway_gene_plots_exact"
os.makedirs(base_output_dir, exist_ok=True)

all_stats = {}
total_genes = sum(len(genes) for genes in pathways.values())
processed_genes = 0

print(f"Starting batch processing of {total_genes} genes across {len(pathways)} pathways...")
print("=" * 60)

for pathway_name, gene_list in pathways.items():
    print(f"\nProcessing pathway: {pathway_name}")
    print(f"Genes: {gene_list}")
    
  
    pathway_output_dir = os.path.join(base_output_dir, pathway_name)
    os.makedirs(pathway_output_dir, exist_ok=True)
    
    pathway_stats = {}
    
    for gene in gene_list:
        processed_genes += 1
        print(f"\n[{processed_genes}/{total_genes}] Processing: {gene}")
        
      
        stats_df = plot_single_gene_points_box_bordered(
            adata=adata,
            gene_name=gene,
            output_dir=pathway_output_dir,
            group_col_ft1="follicle_type1",
            leiden_q1_col="leiden_q1",
            exclude_main={"other"},
            reference_main="healthy",
            use_raw=True,
            p_adjust='none',
            point_size=60,
            jitter_strength=0.5,
            group_sep=0.8,
            tick_fontsize=9,
            title_fontsize=10,
            fig_width=8,
            fig_height=2,
            margin_x=0.01,
            margin_y=0.05,
            save_pdf=True,
            verbose=True
        )
        
        if stats_df is not None:
            pathway_stats[gene] = stats_df
        else:
            print(f"  Warning: No data generated for gene {gene}")
    
    all_stats[pathway_name] = pathway_stats
    success_count = len(pathway_stats)
    print(f"\nCompleted pathway: {pathway_name} ({success_count}/{len(gene_list)} genes processed)")

print("\n" + "=" * 60)
print("BATCH PROCESSING COMPLETED!")
print(f"Results saved in directory: {base_output_dir}/")
print(f"Total pathways processed: {len(all_stats)}")


for pathway_name, pathway_data in all_stats.items():
    print(f"  {pathway_name}: {len(pathway_data)} genes")


print(f"\nGenerating combined statistics file...")
summary_stats = []
for pathway_name, pathway_data in all_stats.items():
    for gene, stats_df in pathway_data.items():
        if stats_df is not None:
            stats_df_copy = stats_df.copy()
            stats_df_copy['pathway'] = pathway_name
            summary_stats.append(stats_df_copy)

if summary_stats:
    combined_stats = pd.concat(summary_stats, ignore_index=True)
    combined_file = os.path.join(base_output_dir, "all_pathways_combined_stats.csv")
    combined_stats.to_csv(combined_file, index=False)
    print(f"Combined statistics saved: {combined_file}")

print(f"\nAll done! Check the '{base_output_dir}' directory for your results.")